# Network Analytics for HR: Who Holds the Organisation Together, and Who Is About to Leave?

**Business problem.** A 600-person company is losing people. HR sees individual survey scores and tenure, but not *how people are connected*. We use **network science** on a collaboration (email/chat) network to answer three questions:

1. **Structure** – Are there silos? Which departments really talk to each other?
2. **Roles** – Who are the *central* connectors and who sits on the *periphery*? Does attrition differ between them?
3. **Risk** – Does attrition spread through the network, and does adding network features improve our flight-risk prediction? And what happens to the organisation if its key connectors leave?

**Same toolkit as the portfolio notebook, new domain**

| Portfolio case study | HR case study |
|---|---|
| Stock | Employee |
| Return correlation between two stocks | Interaction strength between two employees (messages per month) |
| Distance = sqrt(2(1 - corr)) | Distance = 1 / messages per month |
| Minimum Spanning Tree filters the hairball | MST gives the collaboration *backbone* |
| Central vs peripheral stocks | Central vs peripheral employees |
| Portfolio value vs S&P 500 in 2021 | Attrition of each group, and network resilience when people leave |

> **Data is synthetic.** No real employee data is used. `generate_synthetic_hr_data` creates the employees and their interactions with a fixed random seed, and plants realistic effects (isolated people and people whose close colleagues quit are more likely to leave). The *methods* transfer to real HR data, but the *size of the effects* you see here is by construction, not a finding about real companies. On first run the data is saved to `hr_employees.csv` and `hr_interactions.csv`; delete them to regenerate.

> **Privacy note for real use.** Network analytics on employee communication needs consent, aggregation and a clear purpose. Use it to improve how the organisation works, not to monitor individuals.

## 1. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

EMPLOYEES_CSV = Path('hr_employees.csv')
INTERACTIONS_CSV = Path('hr_interactions.csv')
SEED = 42
GROUP_SIZE = 60          # employees in the central / peripheral groups (10% of staff each)

## 2. Synthetic data

**Employees** have a department, level, tenure, location, satisfaction and performance score.
**Interactions** are undirected pairs with a monthly message count. The network is built so that:
- people talk mostly within their department (silos),
- a few *brokers* have many cross-department ties,
- senior people are more connected, remote people less so,
- some people are barely connected at all.

**Attrition happens in two waves** (months 1–6 and months 7–12):
- *Wave 1* depends on the person's own situation: lower satisfaction, short tenure, remote work and few connections raise the risk.
- *Wave 2* depends on the same factors **plus** the share of one's colleagues (weighted by interaction strength) who left in wave 1. This is the contagion effect we later try to detect.

In [ ]:
def generate_synthetic_hr_data(seed=SEED):
    rng = np.random.default_rng(seed)
    dept_sizes = {'Engineering': 170, 'Sales': 130, 'Operations': 110,
                  'Customer Support': 90, 'Finance & HR': 50, 'Marketing': 50}
    dept = np.repeat(list(dept_sizes), list(dept_sizes.values()))
    n = len(dept)
    emp_id = np.array([f'E{i + 1:04d}' for i in range(n)])
    z = lambda x: (x - x.mean()) / x.std()
    sigmoid = lambda x: 1 / (1 + np.exp(-x))

    # ---- employee attributes
    level = rng.choice(['Junior', 'Mid', 'Senior', 'Manager'], size=n, p=[.45, .32, .18, .05])
    tenure = np.clip(rng.gamma(2.0, 2.0, n), 0.2, 20)
    location = rng.choice(['Onsite', 'Hybrid', 'Remote'], size=n, p=[.35, .45, .20])
    satisfaction = np.clip(rng.normal(3.5, 0.8, n), 1, 5)
    performance = np.clip(rng.normal(3.2, 0.7, n), 1, 5)

    # ---- how sociable each person is, and who the cross-department brokers are
    is_broker = rng.random(n) < 0.04
    sociability = rng.lognormal(0, 0.6, n)
    sociability *= np.where(np.isin(level, ['Senior', 'Manager']), 1.4, 1.0)
    sociability *= np.where(location == 'Remote', 0.6, 1.0)
    sociability /= sociability.mean()

    # ---- interaction probabilities: dense inside departments, sparse across
    size = pd.Series(dept).map(dept_sizes).to_numpy()
    same = dept[:, None] == dept[None, :]
    p_within = 8 / (size - 1)               # ~8 within-department contacts on average
    p_cross = 1.2 / (n - size)              # ~1 cross-department contact on average
    base = np.where(same, p_within[:, None], np.sqrt(np.outer(p_cross, p_cross)))
    broker_boost = np.where((is_broker[:, None] | is_broker[None, :]) & ~same, 8.0, 1.0)
    prob = np.minimum(1.0, base * np.sqrt(np.outer(sociability, sociability)) * broker_boost)

    iu = np.triu_indices(n, 1)
    keep = rng.random(len(iu[0])) < prob[iu]
    a, b = iu[0][keep], iu[1][keep]
    msgs = 1 + rng.poisson(6 * np.sqrt(sociability[a] * sociability[b]))

    # ---- make sure everybody is reachable: link stray components to their own department
    g = nx.Graph()
    g.add_nodes_from(range(n))
    g.add_edges_from(zip(a.tolist(), b.tolist()))
    comps = sorted(nx.connected_components(g), key=len, reverse=True)
    extra = []
    for comp in comps[1:]:
        node = next(iter(comp))
        partners = [j for j in comps[0] if dept[j] == dept[node]]
        extra.append((min(node, partners[0]), max(node, partners[0])))
    if extra:
        a = np.concatenate([a, [e[0] for e in extra]])
        b = np.concatenate([b, [e[1] for e in extra]])
        msgs = np.concatenate([msgs, np.ones(len(extra), dtype=int)])

    # ---- attrition in two waves
    degree = np.bincount(np.concatenate([a, b]), minlength=n)
    personal = (-0.9 * z(satisfaction) - 0.35 * z(np.log1p(tenure)) + 0.35 * (location == 'Remote')
                + 0.15 * z(performance) - 0.7 * z(np.log1p(degree)))
    left1 = rng.random(n) < sigmoid(-2.9 + personal)

    W = np.zeros((n, n))
    W[a, b] = msgs
    W[b, a] = msgs
    neighbour_left_share = (W @ left1) / W.sum(axis=1)
    left2 = (~left1) & (rng.random(n) < sigmoid(-3.2 + personal + 4.0 * neighbour_left_share))

    employees = pd.DataFrame({
        'emp_id': emp_id, 'department': dept, 'level': level,
        'tenure_years': tenure.round(1), 'location': location,
        'satisfaction': satisfaction.round(2), 'performance': performance.round(2),
        'left_wave1': left1, 'left_wave2': left2,
    })
    interactions = pd.DataFrame({'emp_a': emp_id[a], 'emp_b': emp_id[b], 'messages_per_month': msgs})
    return employees, interactions


if EMPLOYEES_CSV.exists() and INTERACTIONS_CSV.exists():
    employees = pd.read_csv(EMPLOYEES_CSV)
    interactions = pd.read_csv(INTERACTIONS_CSV)
else:
    employees, interactions = generate_synthetic_hr_data()
    employees.to_csv(EMPLOYEES_CSV, index=False)
    interactions.to_csv(INTERACTIONS_CSV, index=False)

employees = employees.set_index('emp_id')
print(f'{len(employees)} employees, {len(interactions)} interactions')
employees.head()

In [ ]:
employees['left_12m'] = employees['left_wave1'] | employees['left_wave2']
print(f"Left in wave 1: {employees['left_wave1'].mean():.1%} of all employees")
print(f"Left in wave 2: {employees.loc[~employees['left_wave1'], 'left_wave2'].mean():.1%} of wave-1 stayers")
print(f"Left in 12 months: {employees['left_12m'].mean():.1%}")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
employees.groupby('department')['left_12m'].mean().sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('12-month attrition by department')
sns.histplot(employees['satisfaction'], bins=25, ax=axes[1], color='steelblue')
axes[1].set_title('Satisfaction')
sns.histplot(interactions['messages_per_month'], bins=30, ax=axes[2], color='steelblue')
axes[2].set_title('Messages per month per tie');

## 3. The collaboration network and its backbone
Each employee is a node, each pair that exchanges messages is an edge weighted by volume. As in the stock case, we turn the weight into a **distance** (strong ties = short distance) and keep only the **Minimum Spanning Tree**: the N-1 strongest links that still connect everyone.

In [ ]:
G = nx.Graph()
G.add_nodes_from(employees.index)
for r in interactions.itertuples(index=False):
    G.add_edge(r.emp_a, r.emp_b, weight=r.messages_per_month, distance=1 / r.messages_per_month)
nx.set_node_attributes(G, employees['department'].to_dict(), 'department')

mst = nx.minimum_spanning_tree(G, weight='distance')

print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, '
      f'density {nx.density(G):.3f}, connected: {nx.is_connected(G)}')
print(f'Backbone (MST): {mst.number_of_edges()} edges')
print(f'Average shortest path (full network, hops): {nx.average_shortest_path_length(G):.2f}')
print(f'Average shortest path (backbone, hops):     {nx.average_shortest_path_length(mst):.2f}')

In [ ]:
dept_names = sorted(employees['department'].unique())
dept_color = dict(zip(dept_names, sns.color_palette('tab10', len(dept_names))))
node_colors = [dept_color[G.nodes[n]['department']] for n in G.nodes]

pos = nx.spring_layout(G, seed=SEED, k=0.12, iterations=100)

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
nx.draw_networkx(G, pos, ax=axes[0], with_labels=False, node_size=18, node_color=node_colors,
                 edge_color='lightgrey', width=0.3)
axes[0].set_title('Full collaboration network')
nx.draw_networkx(mst, pos, ax=axes[1], with_labels=False, node_size=18, node_color=node_colors,
                 edge_color='grey', width=0.6)
axes[1].set_title('Backbone (minimum spanning tree)')
for ax in axes:
    ax.axis('off')
handles = [plt.Line2D([], [], marker='o', ls='', color=c, label=d) for d, c in dept_color.items()]
axes[0].legend(handles=handles, loc='lower left');

## 4. Silos: how much do departments talk to each other?
We compare the formal structure (departments) with the *informal* structure found by **Louvain community detection**. A high share of ties inside departments, and communities that line up with departments, indicate silos.

In [ ]:
dept_of = employees['department']
edges = interactions.assign(dept_a=interactions['emp_a'].map(dept_of),
                            dept_b=interactions['emp_b'].map(dept_of))
cross_share = (edges['dept_a'] != edges['dept_b']).mean()
print(f'Share of ties that cross departments: {cross_share:.1%}')

mix = pd.crosstab(edges['dept_a'], edges['dept_b'])
mix = mix.add(mix.T, fill_value=0)                       # symmetric edge counts
for d in mix.index:                                      # an edge inside a department is counted twice
    mix.loc[d, d] = mix.loc[d, d] / 2
mix_share = mix.div(mix.sum(axis=1), axis=0)

communities = nx.community.louvain_communities(G, weight='weight', seed=SEED)
communities = sorted(communities, key=len, reverse=True)
dept_partition = [set(employees.index[employees['department'] == d]) for d in dept_names]
print(f'Louvain found {len(communities)} communities; '
      f'modularity {nx.community.modularity(G, communities, weight="weight"):.3f} '
      f'(by department: {nx.community.modularity(G, dept_partition, weight="weight"):.3f})')

community_of = {n: i for i, c in enumerate(communities) for n in c}
employees['community'] = pd.Series(community_of)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.heatmap(mix_share, annot=True, fmt='.0%', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title('Where each department\'s ties go (row = share)')
axes[0].set_xlabel(''); axes[0].set_ylabel('')
sns.heatmap(pd.crosstab(employees['community'], employees['department']), annot=True, fmt='d',
            cmap='Greens', ax=axes[1], cbar=False)
axes[1].set_title('Louvain community vs department (employees)');

## 5. Central and peripheral employees
Same recipe as the stock selection. Each employee gets:

| Criterion | Meaning | Central = |
|---|---|---|
| Degree centrality | share of colleagues directly connected | high |
| Betweenness centrality | how often they lie on the strongest-path routes between others (shortest paths by *distance*) | high |
| Distance on degree | hops *on the backbone* to the best-connected employee | low |
| Distance to the core | hops on the backbone to the employee with the smallest average hop distance to everybody | low |

*Central score* = mean percentile rank of degree and betweenness. *Peripheral score* = mean of the two backbone hop distances.

In [ ]:
node_stats = pd.DataFrame({
    'degree_centrality': nx.degree_centrality(G),
    'strength': dict(G.degree(weight='weight')),
    'betweenness': nx.betweenness_centrality(G, weight='distance'),
    'clustering': nx.clustering(G),
})
node_stats['cross_dept_share'] = pd.Series({
    n: np.mean([dept_of[m] != dept_of[n] for m in G[n]]) for n in G.nodes})

hops = dict(nx.all_pairs_shortest_path_length(mst))
hub_degree = node_stats['degree_centrality'].idxmax()
mean_hops = pd.Series({n: np.mean([d for m, d in hops[n].items() if m != n]) for n in mst.nodes})
core = mean_hops.idxmin()
print(f'Best-connected employee: {hub_degree} ({dept_of[hub_degree]}) | '
      f'backbone core: {core} ({dept_of[core]})')

node_stats['hops_to_hub'] = pd.Series({n: hops[n][hub_degree] for n in mst.nodes})
node_stats['hops_to_core'] = pd.Series({n: hops[n][core] for n in mst.nodes})
node_stats['central_score'] = node_stats[['degree_centrality', 'betweenness']].rank(pct=True).mean(axis=1)
node_stats['peripheral_score'] = node_stats[['hops_to_hub', 'hops_to_core']].mean(axis=1)

central_group = node_stats.nlargest(GROUP_SIZE, 'central_score').index
peripheral_group = node_stats.nlargest(GROUP_SIZE, 'peripheral_score').index
print('Overlap between groups:', len(set(central_group) & set(peripheral_group)))

employees = employees.join(node_stats)
employees['group'] = 'Other'
employees.loc[central_group, 'group'] = 'Central'
employees.loc[peripheral_group, 'group'] = 'Peripheral'
employees.loc[central_group].groupby('department').size().to_frame('central employees').T

In [ ]:
color_map = {'Central': 'red', 'Peripheral': 'green', 'Other': 'lightblue'}
fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx(mst, pos, ax=ax, with_labels=False, edge_color='lightgrey', width=0.5,
                 node_color=[color_map[employees.loc[n, 'group']] for n in mst.nodes],
                 node_size=[60 if employees.loc[n, 'group'] != 'Other' else 12 for n in mst.nodes])
ax.set_title('Backbone: central (red) vs peripheral (green) employees'); ax.axis('off');

## 6. Does attrition differ between central and peripheral employees?
This is the analogue of comparing the central and peripheral portfolios.

In [ ]:
def attrition_table(df, by):
    rows = df.groupby(by).agg(
        employees=('left_12m', 'size'),
        left_wave1=('left_wave1', 'mean'),
        left_12m=('left_12m', 'mean'),
        avg_satisfaction=('satisfaction', 'mean'),
        avg_tenure=('tenure_years', 'mean'),
    )
    stayers = df[~df['left_wave1']].groupby(by)['left_wave2'].mean().rename('left_wave2_of_stayers')
    return rows.join(stayers)

group_table = attrition_table(employees, 'group').loc[['Central', 'Other', 'Peripheral']]
display_cols = ['left_wave1', 'left_wave2_of_stayers', 'left_12m']
print((group_table[display_cols] * 100).round(1).join(group_table[['employees', 'avg_satisfaction', 'avg_tenure']].round(2)))

ax = (group_table[display_cols] * 100).plot.bar(figsize=(10, 5), rot=0, color=['#4c78a8', '#f58518', '#54a24b'])
ax.set(ylabel='% of group', xlabel='', title='Attrition by network position')
ax.legend(['Wave 1 (all)', 'Wave 2 (of stayers)', '12 months (all)']);

In [ ]:
employees['degree_quartile'] = pd.qcut(employees['degree_centrality'], 4,
                                       labels=['Q1 (fewest ties)', 'Q2', 'Q3', 'Q4 (most ties)'])
quartiles = attrition_table(employees, 'degree_quartile')
print((quartiles[['left_wave1', 'left_12m']] * 100).round(1))
(quartiles['left_12m'] * 100).plot.bar(figsize=(8, 4), rot=0, color='steelblue',
                                       title='12-month attrition by number of ties (quartile)');

## 7. Is attrition contagious?
Take everyone who **stayed through wave 1** and has at least 5 ties, and measure what share of their colleagues (weighted by interaction volume) left in wave 1. If leaving spreads through ties, wave-2 attrition should rise with that share.

In [ ]:
left1 = employees['left_wave1'].astype(float)
employees['neighbour_left_share'] = pd.Series({
    n: sum(G[n][m]['weight'] * left1[m] for m in G[n]) / sum(G[n][m]['weight'] for m in G[n])
    for n in G.nodes})

# Only people with at least 5 ties: with very few ties the share is mostly 0% or 100% and mixes up
# "isolated" with "exposed to leavers".
n_ties = employees['degree_centrality'] * (len(G) - 1)
stayers = employees[(~employees['left_wave1']) & (n_ties >= 5)].copy()
stayers['exposure'] = pd.cut(stayers['neighbour_left_share'], [-0.01, 0, 0.1, 0.25, 1],
                             labels=['None', 'Low (0-10%)', 'Medium (10-25%)', 'High (>25%)'])
exposure = stayers.groupby('exposure').agg(employees=('left_wave2', 'size'), wave2_attrition=('left_wave2', 'mean'))
exposure['wave2_attrition'] = (exposure['wave2_attrition'] * 100).round(1)
print(exposure)

ax = exposure['wave2_attrition'].plot.bar(figsize=(8, 4), rot=0, color='indianred')
ax.set(xlabel='Share of colleagues who left in wave 1', ylabel='Wave-2 attrition, %',
       title='Attrition rises with exposure to leavers');

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
status = ['red' if employees.loc[n, 'left_wave1'] else 'orange' if employees.loc[n, 'left_wave2'] else 'lightblue'
          for n in G.nodes]
nx.draw_networkx(G, pos, ax=ax, with_labels=False, edge_color='#eeeeee', width=0.3, node_color=status,
                 node_size=[40 if s != 'lightblue' else 8 for s in status])
ax.set_title('Leavers: wave 1 (red), wave 2 (orange). Leavers cluster in neighbourhoods.'); ax.axis('off');

## 8. Does the network improve flight-risk prediction?
Predict **wave-2 leavers among wave-1 stayers**, using only information available at the end of wave 1:
- *Individual model*: satisfaction, tenure, performance, remote flag.
- *Individual + network model*: the above plus number of ties, betweenness, cross-department share and the share of colleagues who already left.

Evaluation is 5-fold cross-validated AUC (repeated with different splits). With only a few dozen leavers the numbers are noisy, so treat the gap as indicative.

In [ ]:
stayers = employees[~employees['left_wave1']].copy()
stayers['remote'] = (stayers['location'] == 'Remote').astype(int)
stayers['log_tenure'] = np.log1p(stayers['tenure_years'])
stayers['log_ties'] = np.log1p(stayers['degree_centrality'] * (len(G) - 1))
y = stayers['left_wave2'].astype(int)

individual = ['satisfaction', 'log_tenure', 'performance', 'remote']
network = ['log_ties', 'betweenness', 'cross_dept_share', 'neighbour_left_share']

def cv_auc(features, repeats=10):
    aucs, oof_mean = [], np.zeros(len(stayers))
    for r in range(repeats):
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        proba = cross_val_predict(model, stayers[features], y,
                                  cv=StratifiedKFold(5, shuffle=True, random_state=r), method='predict_proba')[:, 1]
        aucs.append(roc_auc_score(y, proba))
        oof_mean += proba / repeats
    return np.mean(aucs), np.std(aucs), oof_mean

auc_ind, sd_ind, _ = cv_auc(individual)
auc_net, sd_net, oof = cv_auc(individual + network)
print(f'Wave-2 leavers: {y.sum()} of {len(y)} stayers ({y.mean():.1%})')
print(f'AUC individual only:      {auc_ind:.3f} (+/- {sd_ind:.3f})')
print(f'AUC individual + network: {auc_net:.3f} (+/- {sd_net:.3f})')

top = pd.Series(oof, index=stayers.index).nlargest(int(0.2 * len(stayers))).index
print(f'Flagging the top 20% highest-risk employees captures {y[top].sum() / y.sum():.0%} of wave-2 leavers.')

pd.Series({'Individual': auc_ind, 'Individual + network': auc_net}).plot.bar(
    figsize=(6, 4), rot=0, color=['grey', 'steelblue'], ylim=(0.5, 1), title='Cross-validated AUC');

## 9. Resilience: what if the key connectors leave?
The portfolio notebook asked how each basket performs. Here we ask how the **organisation** performs when people leave. We remove 5–25% of employees using four strategies and track:
- the share of remaining employees still in one connected network (information can still flow),
- the average number of hops between employees in that network (how far information has to travel).

A network that falls apart when its central people leave depends on them.

In [ ]:
def network_health(graph, removed):
    h = graph.copy()
    h.remove_nodes_from(removed)
    giant = h.subgraph(max(nx.connected_components(h), key=len))
    return len(giant) / h.number_of_nodes(), nx.average_shortest_path_length(giant)

order = {
    'Central first': node_stats.sort_values('central_score', ascending=False).index.tolist(),
    'Highest betweenness first': node_stats.sort_values('betweenness', ascending=False).index.tolist(),
    'Peripheral first': node_stats.sort_values('peripheral_score', ascending=False).index.tolist(),
}
fractions = [0, 0.05, 0.10, 0.15, 0.20, 0.25]
rng = np.random.default_rng(SEED)
rows = []
for f in fractions:
    k = int(round(f * len(G)))
    for name, ranking in order.items():
        connected, path = network_health(G, ranking[:k])
        rows.append((name, f, connected, path))
    randoms = [network_health(G, rng.choice(list(G.nodes), k, replace=False)) for _ in range(5)]
    rows.append(('Random', f, np.mean([r[0] for r in randoms]), np.mean([r[1] for r in randoms])))

resilience = pd.DataFrame(rows, columns=['strategy', 'fraction_removed', 'connected_share', 'avg_path_length'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=resilience, x='fraction_removed', y='connected_share', hue='strategy', marker='o', ax=axes[0])
axes[0].set(title='Share of remaining staff in the main network', xlabel='Share of employees leaving', ylabel='')
sns.lineplot(data=resilience, x='fraction_removed', y='avg_path_length', hue='strategy', marker='o', ax=axes[1])
axes[1].set(title='Average hops between employees', xlabel='Share of employees leaving', ylabel='')
resilience[resilience['fraction_removed'] == 0.25].set_index('strategy').round(3)

## 10. Summary

In [ ]:
central = group_table.loc['Central']
peripheral = group_table.loc['Peripheral']
other = group_table.loc['Other']
high_exposure = exposure['wave2_attrition'].iloc[-1]
no_exposure = exposure['wave2_attrition'].iloc[0]
base_path = resilience.loc[resilience['fraction_removed'] == 0, 'avg_path_length'].iloc[0]
at25 = resilience[resilience['fraction_removed'] == 0.25].set_index('strategy')

print(f'''KEY FINDINGS (synthetic data)

1. Silos: {1 - cross_share:.0%} of ties stay inside a department; only {cross_share:.0%} cross departments.
2. Roles: 12-month attrition is {central['left_12m']:.0%} for central employees, {other['left_12m']:.0%} for everyone else
   and {peripheral['left_12m']:.0%} for peripheral employees.
3. Contagion: stayers whose colleagues left have wave-2 attrition of {high_exposure:.0f}% (high exposure)
   vs {no_exposure:.0f}% (none).
4. Prediction: adding network features moves AUC from {auc_ind:.2f} to {auc_net:.2f}.
5. Resilience: if 25% of staff leave, the average distance between employees grows from {base_path:.2f} hops to
   {at25.loc['Central first', 'avg_path_length']:.2f} when central people go first, versus {at25.loc['Random', 'avg_path_length']:.2f}
   when people leave at random. {at25.loc['Central first', 'connected_share']:.0%} of the rest stay connected in one network.''')

### What an HR team would do with this
- **Retention outreach for the periphery.** Isolated people leave more; pair them with buddies, mentors or cross-team projects *before* they disengage.
- **Watch exposure, not only individuals.** When someone leaves, check the people they worked with most closely and talk to them early.
- **Protect and reinforce brokers and central connectors.** They hold the organisation together; plan succession and spread their knowledge.
- **Break silos deliberately.** Create cross-department ties where the mix matrix is thin.

### Caveats for real data
- Effects here are planted; real effects are usually weaker and noisier. Validate on your own history.
- Correlation is not causation: colleagues may leave together for a shared reason (a bad manager, a reorganisation) rather than through influence.
- Communication metadata is sensitive. Aggregate, de-identify and be transparent about purpose.